In [0]:
from pyspark.sql.functions import *

store_df = spark.read.table("fk_ce_2026_07_1382.bronze.stores")
standardized_orders_df = (
    store_df
    .drop("file_path", "ingestion_timestamp")
    .withColumns({
        "store_id": upper(trim(col("store_id"))),
        "store_name": trim(col("store_name")),
        "city": trim(col("city")),
        "state": trim(col("state")),
        "region": trim(col("region")),

        "opening_date": try_to_date(col("opening_date"), "yyyy-MM-dd")
    })
)

standardized_orders_df.write.mode("overwrite").saveAsTable("fk_ce_2026_07_1382.silver.stores")

In [0]:
from pyspark.sql.functions import *

df_products = spark.read.table("fk_ce_2026_07_1382.bronze.products")
standardized_products_df = (
    df_products
    .withColumns({
        "product_id": upper(trim(col("product_id"))),
        "product_name": trim(col("product_name")),
        "brand": trim(col("brand")),

        "category_l1": col("category.l1"),
        "category_l2": col("category.l2"),

        "pack_size": col("pack.size").cast("int"),
        "pack_unit": col("pack.unit"),

        "mrp": col("mrp").cast("decimal(10,2)"),
        "selling_price": col("selling_price").cast("decimal(10,2)"),

        "last_changed_snapshot_date": try_to_date(col("snapshot_date"), "yyyy-MM-dd")
    })
    .drop("file_path", "ingestion_timestamp", "category", "pack")
)

In [0]:
standardized_products_df.createOrReplaceTempView("standardized_products")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS fk_ce_2026_07_1382.silver.products (
  product_id STRING,
  product_name STRING,
  brand STRING,
  is_active BOOLEAN,
  mrp DECIMAL(10,2),
  selling_price DECIMAL(10,2),
  snapshot_date STRING,
  tags ARRAY<STRING>,
  category_l1 STRING,
  category_l2 STRING,
  pack_size INT,
  pack_unit STRING,
  last_changed_snapshot_date DATE
)

In [0]:
%sql
MERGE INTO fk_ce_2026_07_1382.silver.products AS t
USING standardized_products AS s
ON t.product_id = s.product_id
WHEN MATCHED THEN
  UPDATE SET * 
WHEN NOT MATCHED THEN
  INSERT *;

In [0]:
from pyspark.sql.functions import *

df_products = spark.read.table("fk_ce_2026_07_1382.bronze.customers")
standardized_products_df = (
    df_products
    .drop("file_path", "ingestion_timestamp")
    .withColumns({
        "customer_id": upper(trim(col("customer_id"))),
        "full_name": trim(col("full_name")),
        "email": trim(col("email")),
        "phone": trim(col("phone")),
        "city": trim(col("city")),
        "tier": trim(col("tier")),
        "signup_date": try_to_date(col("signup_date"), "yyyy-MM-dd")
    })
    .dropDuplicates()
)
standardized_products_df.write.mode("append").saveAsTable("fk_ce_2026_07_1382.silver.customers_staging")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS fk_ce_2026_07_1382.silver.customers (
  customer_id STRING,
  full_name STRING,
  email STRING,
  phone STRING,
  city STRING,
  tier STRING,
  signup_date DATE,
  effective_from DATE,
  effective_to DATE,
  is_current BOOLEAN
)

In [0]:
%sql
-- SCD Type 2 target table with tracking columns
CREATE OR REPLACE TABLE fk_ce_2026_07_1382.silver.customers (
  customer_id      STRING,
  full_name        STRING,
  email            STRING,
  phone            STRING,
  city             STRING,
  tier             STRING,
  signup_date      DATE,
  effective_from   DATE,   
  effective_to     DATE,   
  is_current       BOOLEAN 
);

-- SCD Type 2 MERGE: expire old versions, insert new / changed versions
MERGE INTO fk_ce_2026_07_1382.silver.customers AS t
USING (
  -- Part 1: Expire old current records whose attributes have changed
  SELECT
    tc.customer_id,
    tc.full_name,
    tc.email,
    tc.phone,
    tc.city,
    tc.tier,
    tc.signup_date,
    tc.effective_from,                      
    current_date()        AS effective_to,    
    false                 AS is_current
  FROM fk_ce_2026_07_1382.silver.customers tc
  JOIN fk_ce_2026_07_1382.silver.customers_staging s
    ON tc.customer_id = s.customer_id
   AND tc.is_current  = true
  WHERE tc.full_name   IS DISTINCT FROM s.full_name
     OR tc.email       IS DISTINCT FROM s.email
     OR tc.phone       IS DISTINCT FROM s.phone
     OR tc.city        IS DISTINCT FROM s.city
     OR tc.tier        IS DISTINCT FROM s.tier
     OR tc.signup_date IS DISTINCT FROM s.signup_date

  UNION ALL

  -- Part 2: New current records (brand-new customers + new versions of changed customers)
  SELECT
    s.customer_id,
    s.full_name,
    s.email,
    s.phone,
    s.city,
    s.tier,
    s.signup_date,
    CASE
      WHEN tc.customer_id IS NULL THEN COALESCE(s.signup_date, current_date())  
      ELSE current_date()                                               
    END                  AS effective_from,
    CAST(NULL AS DATE)   AS effective_to,
    true                 AS is_current
  FROM fk_ce_2026_07_1382.silver.customers_staging s
  LEFT JOIN fk_ce_2026_07_1382.silver.customers tc
    ON s.customer_id = tc.customer_id
   AND tc.is_current  = true
  WHERE tc.customer_id IS NULL                             
     OR tc.full_name   IS DISTINCT FROM s.full_name         
     OR tc.email       IS DISTINCT FROM s.email
     OR tc.phone       IS DISTINCT FROM s.phone
     OR tc.city        IS DISTINCT FROM s.city
     OR tc.tier        IS DISTINCT FROM s.tier
     OR tc.signup_date IS DISTINCT FROM s.signup_date
) AS s
ON    t.customer_id    = s.customer_id
  AND t.effective_from = s.effective_from
  AND t.is_current     = true
WHEN MATCHED AND s.is_current = false THEN
  UPDATE SET t.effective_to = s.effective_to,
              t.is_current   = false
WHEN NOT MATCHED AND s.is_current = true THEN
  INSERT (customer_id, full_name, email, phone, city, tier, signup_date,
          effective_from, effective_to, is_current)
  VALUES (s.customer_id, s.full_name, s.email, s.phone, s.city, s.tier, s.signup_date,
          s.effective_from, s.effective_to, s.is_current);

In [0]:
%sql
select * from fk_ce_2026_07_1382.bronze.orders

In [0]:
from pyspark.sql.functions import *

df_orders = spark.read.table("fk_ce_2026_07_1382.bronze.orders")
standardized_orders_df = (
    df_orders
    .withColumn("item", explode(col("items")))
    .withColumns({
        "order_id": upper(trim(col("order_id"))),
        "customer_id": upper(trim(col("customer_id"))),
        "store_id": upper(trim(col("store_id"))),
        "product_id": upper(trim(col("item.product_id"))),

        "order_status": when(upper(trim(col("order_status"))) == "CANCELED", lit("CANCELLED")).otherwise(upper(trim(col("order_status")))),
        
        "payment_method": upper(trim(col("payment.method"))),

        "order_ts": coalesce(
            try_to_timestamp(col("order_ts"), lit("yyyy-MM-dd HH:mm:ss")),
            try_to_timestamp(col("order_ts"), lit("MM/dd/yyyy'T'HH:mm:ss"))
        ),

        "updated_at": coalesce(
            try_to_timestamp(col("updated_at"), lit("yyyy-MM-dd HH:mm:ss"))
        ),

        "quantity": expr("try_cast(item.qty as int)"), 

        "discount_amount": col("item.discount")
    })
    .filter(col("order_status").isin(["PLACED", "DELIVERED", "CANCELLED"]))
    .filter((col("quantity") > 0) & (col("quantity") % 1 == 0))
    .filter(col("item.unit_price") > 0)

    .fillna({"discount_amount": 0})
)


In [0]:
standardized_orders_df.createOrReplaceTempView("standardized_orders_df")

In [0]:
%sql
CREATE OR REPLACE TABLE fk_ce_2026_07_1382.silver.deduplicated_orders AS
WITH RankedOrders AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY order_id 
            ORDER BY updated_at DESC
        ) AS rn
    FROM standardized_orders_df
)
SELECT 
    *
FROM RankedOrders
WHERE rn = 1;

In [0]:
%sql
select * from fk_ce_2026_07_1382.silver.deduplicated_orders

In [0]:
%sql
-- Build a validation view that flags each order with a rejection reason (or NULL if valid)
CREATE OR REPLACE TEMP VIEW validated_orders AS
SELECT
    d.*,
    CASE
        WHEN d.customer_id IS NULL THEN 'missing_customer'
        WHEN NOT EXISTS (
            SELECT 1 FROM fk_ce_2026_07_1382.silver.customers c
            WHERE c.customer_id = d.customer_id
        ) THEN 'unknown_customer'
        WHEN d.store_id IS NULL OR NOT EXISTS (
            SELECT 1 FROM fk_ce_2026_07_1382.silver.stores s
            WHERE s.store_id = d.store_id
        ) THEN 'unknown_store'
        WHEN d.order_ts IS NULL THEN 'invalid_timestamp'
        WHEN d.items IS NULL OR size(d.items) = 0 THEN 'empty_order'
        WHEN d.product_id IS NULL OR NOT EXISTS (
            SELECT 1 FROM fk_ce_2026_07_1382.silver.products p
            WHERE p.product_id = d.product_id
        ) THEN 'product_missing'
        WHEN d.quantity < 0 OR d.quantity != cast(d.quantity AS int) THEN 'invalid_quantity'
        WHEN d.item.unit_price < 0 THEN 'invalid_price'
        ELSE NULL
    END AS rejection_region
FROM fk_ce_2026_07_1382.silver.deduplicated_orders d;

-- Quarantine table: rejected orders
CREATE OR REPLACE TABLE fk_ce_2026_07_1382.silver.quarantine_orders AS
SELECT
    order_id,
    rejection_region,
    updated_at,
    to_json(struct(*)) AS raw_records,
    current_timestamp() AS quarantined_at
FROM validated_orders
WHERE rejection_region IS NOT NULL;

-- Valid orders table: orders that passed all validation rules
CREATE OR REPLACE TABLE fk_ce_2026_07_1382.silver.orders AS
SELECT
    order_id,
    customer_id,
    store_id,
    product_id,
    order_ts,
    date(order_ts) AS order_date,
    order_status,
    payment_method,
    delivery.promised_mins AS promised_minutes,
    delivery.actual_mins AS actual_minutes,
    delivery.delivery_fee AS delivery_fee,
    updated_at
FROM validated_orders
WHERE rejection_region IS NULL;

In [0]:
%sql
CREATE OR REPLACE TABLE fk_ce_2026_07_1382.silver.order_items AS
SELECT
    order_id,
    product_id,
    quantity,
    item.unit_price AS unit_price,
    discount_amount,
    quantity * unit_price AS gross_amount,
    gross_amount - discount_amount AS net_amount
FROM validated_orders

In [0]:
%sql
select * from fk_ce_2026_07_1382.silver.order_items